In [1]:
import torch
print(torch.backends.mps.is_available())  # should print True
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

True
Using device: mps


In [2]:
import torch
print(torch.backends.mps.is_available())  # should print True
print(torch.device("mps"))                # should print mps

True
mps


In [3]:
import pandas as pd
import numpy as np
import os
import json
import requests
import zipfile
from sklearn.model_selection import train_test_split

print("All imports OK")

All imports OK


In [5]:
from datasets import load_dataset

os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

dataset = load_dataset("christinacdl/clickbait_notclickbait_dataset")
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 43802
    })
    validation: Dataset({
        features: ['label', 'text'],
        num_rows: 2191
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 8760
    })
})
{'label': 0, 'text': 'Alphabet Scraps Plan to Blanket Globe With Internet Balloons'}


In [6]:
df = pd.DataFrame(dataset["train"])

# Rename columns to match our project
df = df.rename(columns={"text": "headline", "label": "label"})

print(df.shape)
print(df.head())
print("\nLabel distribution:")
print(df["label"].value_counts())

(43802, 2)
   label                                           headline
0      0  Alphabet Scraps Plan to Blanket Globe With Int...
1      0  US Boy Scouts and hikers airlifted from wildfi...
2      1  Here's What Happened When I Road Tripped Aroun...
3      1  The Winners And Losers Of Asian Representation...
4      1  Which Disney Channel Show Do You Actually Like...

Label distribution:
label
0    27545
1    16257
Name: count, dtype: int64


In [7]:
# Drop missing headlines
df = df.dropna(subset=["headline"])

# Remove duplicates
df = df.drop_duplicates(subset=["headline"])

# Strip whitespace
df["headline"] = df["headline"].str.strip()

# Remove very short headlines (less than 5 words)
df = df[df["headline"].str.split().str.len() >= 5]

print(f"Clean dataset size: {len(df)}")
print(df["label"].value_counts())

Clean dataset size: 42772
label
0    26759
1    16013
Name: count, dtype: int64


In [8]:
# 80% train, 10% val, 10% test
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

train_df.to_csv("../data/processed/train.csv", index=False)
val_df.to_csv("../data/processed/val.csv",   index=False)
test_df.to_csv("../data/processed/test.csv",  index=False)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 34217 | Val: 4277 | Test: 4278


In [9]:
train = pd.read_csv("../data/processed/train.csv")

print("Sample clickbait headlines:")
print(train[train["label"]==1]["headline"].sample(3).tolist())

print("\nSample non-clickbait headlines:")
print(train[train["label"]==0]["headline"].sample(3).tolist())

Sample clickbait headlines:
['Can You Guess The Hogwarts House Of These "Harry Potter" Characters', "Why You Need To Stop What You're Doing And Date A Robot", '28 Tweets About Weddings That Will Make You Laugh Out Loud']

Sample non-clickbait headlines:
['Soon, Cars May Take Away the Keys of a Drunken Driver', 'Australia Moves Cautiously in Effort to Free Executive Detained in China', '25 Cheat Sheets That Make Cooking Healthier Less Of A Freaking Chore']
